# 06 — Comparativa estadística y coste computacional

**Proyecto RESPIR-AI (TFG)** — Comparativa de XGBoost/skforecast, Chronos-2 y TinyTimeMixer (TTM) para la predicción horaria de la tasa de consumo de oxígeno (OUR) en una EDAR.

Este notebook consolida los resultados de los notebooks 03 (baseline XGBoost), 04 (Chronos-2) y 05 (TTM), todos ejecutados sobre el **protocolo de evaluación v2**: *rolling-origin* con 36 orígenes en el conjunto de test, horizontes H ∈ {6, 12, 24, 48} horas y las mismas ventanas exactas para todos los modelos.

Contenido:
1. Tabla consolidada modelo × escenario × H (MAE, RMSE, MAPE, R²).
2. Mejor configuración por familia y horizonte.
3. Test de Diebold–Mariano (corrección de Harvey–Leybourne–Newbold) por pares de mejores configuraciones y por horizonte.
4. Test de Friedman + post-hoc de Nemenyi.
5. Análisis de coste computacional (parámetros, VRAM, tiempos de entrenamiento e inferencia).

> **Nota sobre potencia estadística**: con n = 36 ventanas la potencia de los contrastes es limitada; diferencias de MAE del orden de 0,1–0,2 mg O₂/(L·h) pueden no alcanzar significación aunque sean sistemáticas. Se informa de ello explícitamente en la interpretación.

In [1]:
# Raíz del repositorio como cwd (permite ejecutar desde notebooks/ o desde la raíz)
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/"common_eval.py").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import itertools, math, os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats

# Rutas relativas a la raíz del repositorio
P_BASE = "results/resultados_baseline.csv"
P_CHR = "results/resultados_chronos2.csv"
P_TTM = "results/resultados_ttm.csv"
P_ERR = "results/errores_por_ventana.parquet"
P_COST = "results/coste_computacional.csv"

os.makedirs("results", exist_ok=True)
HS = [6, 12, 24, 48]
FAM = {"xgboost": "XGBoost", "chronos2": "Chronos-2", "ttm": "TTM"}

base = pd.read_csv(P_BASE); chro = pd.read_csv(P_CHR); ttm = pd.read_csv(P_TTM)
err  = pq.read_table(P_ERR).to_pandas()      # MAE por modelo × escenario × H × ventana (36 ventanas)
coste = pd.read_csv(P_COST)
print(base.shape, chro.shape, ttm.shape, err.shape, coste.shape)

(12, 11) (24, 10) (16, 10) (1872, 6) (13, 8)


## 1. Tabla consolidada modelo × escenario × H

Se unen los tres CSV de resultados en una única tabla con las métricas agregadas (*pooled* sobre todos los puntos de las 36 ventanas): MAE, RMSE, MAPE y R².

In [2]:
cons = pd.concat([base, chro, ttm], ignore_index=True)
cons["familia"] = cons["modelo"].map(FAM)
cols = ["familia", "modelo", "escenario", "H", "MAE", "RMSE", "MAPE", "R2",
        "t_inferencia_s", "contexto", "covariables"]
cons = cons[cols].sort_values(["familia", "escenario", "H"]).reset_index(drop=True)
cons.to_csv("results/tabla_consolidada.csv", index=False)
cons.round(3)

Out[0]: 
      familia  ...                                       covariables
0   Chronos-2  ...                                           ninguna
1   Chronos-2  ...                                           ninguna
2   Chronos-2  ...                                           ninguna
3   Chronos-2  ...                                           ninguna
4   Chronos-2  ...                                   proceso pasadas
5   Chronos-2  ...                                   proceso pasadas
6   Chronos-2  ...                                   proceso pasadas
7   Chronos-2  ...                                   proceso pasadas
8   Chronos-2  ...            proceso pasadas + meteo futuras reales
9   Chronos-2  ...            proceso pasadas + meteo futuras reales
10  Chronos-2  ...            proceso pasadas + meteo futuras reales
11  Chronos-2  ...            proceso pasadas + meteo futuras reales
12  Chronos-2  ...                          ninguna (LoRA 500 pasos)
13  Chronos-2  ...       

## 2. Mejor configuración por familia y horizonte

Para cada familia (XGBoost, Chronos-2, TTM) y cada H se selecciona el escenario con menor MAE. Esta selección define los pares que se contrastan estadísticamente en las secciones 3 y 4.

In [3]:
idx = cons.groupby(["familia", "H"])["MAE"].idxmin()
mejor = cons.loc[idx].sort_values(["H", "MAE"]).reset_index(drop=True)
mejor.to_csv("results/mejor_por_familia.csv", index=False)
mejor[["familia", "escenario", "H", "MAE", "RMSE", "MAPE", "R2"]].round(3)

Out[0]: 
      familia                escenario   H    MAE   RMSE    MAPE     R2
0     XGBoost           E1_univariante   6  3.428  4.458  16.981  0.355
1         TTM   E2_ft_univariante_5pct   6  3.575  4.573  17.476  0.321
2   Chronos-2        E1_zs_univariante   6  3.631  4.719  17.431  0.277
3     XGBoost           E1_univariante  12  3.730  4.952  17.863  0.358
4         TTM  E4_ft_exog_futuras_5pct  12  3.763  4.949  17.638  0.359
5   Chronos-2        E4_ft_univariante  12  3.949  5.118  18.102  0.315
6         TTM  E4_ft_exog_futuras_5pct  24  3.810  4.899  17.555  0.291
7     XGBoost           E2_meteo_naive  24  3.864  5.004  18.218  0.260
8   Chronos-2      E3_zs_multi_futuras  24  3.976  5.022  18.128  0.255
9   Chronos-2      E3_zs_multi_futuras  48  4.198  5.244  19.158  0.222
10        TTM  E4_ft_exog_futuras_5pct  48  4.285  5.326  19.568  0.197
11    XGBoost           E2_meteo_naive  48  4.312  5.432  20.339  0.165


## 3. Test de Diebold–Mariano con corrección HLN

El test de Diebold–Mariano (DM) contrasta la hipótesis nula de igual capacidad predictiva entre dos modelos a partir de la serie de diferencias de pérdida $d_t = L_{A,t} - L_{B,t}$. Aquí la pérdida por ventana es el **MAE de la ventana** (una observación por origen; n = 36).

- La varianza de $\bar d$ se estima con un estimador HAC (Newey–West truncado en $h-1$).
- Se aplica la **corrección de Harvey–Leybourne–Newbold (1997)** para muestras pequeñas y el p-valor se calcula con una t de Student con $n-1$ grados de libertad.
- Los orígenes están separados 24 h, de modo que solo a H = 48 hay solapamiento entre ventanas adyacentes; se usa $h = \lceil H/24 \rceil$ (1 para H ≤ 24; 2 para H = 48).

**Limitación**: con 36 ventanas la potencia es reducida; la ausencia de significación no debe interpretarse como evidencia de equivalencia.

In [4]:
def dm_hln(d, h=1):
    """Estadístico DM con corrección HLN y p-valor bilateral (t de Student, n-1 gl).
    d: array de diferencias de pérdida por ventana; h: truncamiento HAC (h-1 retardos)."""
    d = np.asarray(d, float); n = len(d); dbar = d.mean()
    gamma0 = np.mean((d - dbar) ** 2)
    var = gamma0
    for k in range(1, h):
        gk = np.mean((d[k:] - dbar) * (d[:-k] - dbar))
        var += 2 * gk
    if var <= 0:
        var = gamma0                      # ajuste habitual si la HAC no es positiva
    dm = dbar / math.sqrt(var / n)
    hln = math.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)   # corrección HLN
    dm_c = dm * hln
    p = 2 * stats.t.sf(abs(dm_c), df=n - 1)
    return dm_c, p

best_cfg = {(r.familia, r.H): (r.modelo, r.escenario) for r in mejor.itertuples()}
rows = []
for H in HS:
    series = {}
    for fam in ["XGBoost", "Chronos-2", "TTM"]:
        m, esc = best_cfg[(fam, H)]
        s = err[(err.modelo == m) & (err.escenario == esc) & (err.H == H)] \
              .sort_values("ventana")["MAE"].values
        assert len(s) == 36
        series[fam] = (s, esc)
    h = max(1, math.ceil(H / 24))         # 1 para H<=24, 2 para H=48 (solapamiento)
    for a, b in itertools.combinations(["XGBoost", "Chronos-2", "TTM"], 2):
        d = series[a][0] - series[b][0]
        dm_c, p = dm_hln(d, h=h)
        rows.append(dict(H=H, modelo_A=a, config_A=series[a][1],
                         modelo_B=b, config_B=series[b][1],
                         MAE_A=series[a][0].mean(), MAE_B=series[b][0].mean(),
                         dif_media=d.mean(), DM_HLN=dm_c, p_valor=p,
                         h_lag=h, n_ventanas=36))
dm_df = pd.DataFrame(rows)
dm_df.to_csv("results/diebold_mariano.csv", index=False)
dm_df.round(4)

Out[0]: 
     H   modelo_A             config_A  ... p_valor h_lag  n_ventanas
0    6    XGBoost       E1_univariante  ...  0.1338     1          36
1    6    XGBoost       E1_univariante  ...  0.2318     1          36
2    6  Chronos-2    E1_zs_univariante  ...  0.6193     1          36
3   12    XGBoost       E1_univariante  ...  0.1407     1          36
4   12    XGBoost       E1_univariante  ...  0.7794     1          36
5   12  Chronos-2    E4_ft_univariante  ...  0.0695     1          36
6   24    XGBoost       E2_meteo_naive  ...  0.3803     1          36
7   24    XGBoost       E2_meteo_naive  ...  0.6130     1          36
8   24  Chronos-2  E3_zs_multi_futuras  ...  0.1543     1          36
9   48    XGBoost       E2_meteo_naive  ...  0.5359     2          36
10  48    XGBoost       E2_meteo_naive  ...  0.8578     2          36
11  48  Chronos-2  E3_zs_multi_futuras  ...  0.5803     2          36

[12 rows x 12 columns]


**Lectura**: ningún par de mejores configuraciones difiere significativamente al 5 % en ningún horizonte (p mínimo ≈ 0,07, Chronos-2 vs. TTM a H = 12, favorable a TTM). La ventaja de XGBoost a H = 6 (Δ ≈ −0,20 frente a Chronos-2) y la de Chronos-2 con meteorología futura a H = 48 (Δ ≈ −0,11 frente a XGBoost) son consistentes en signo pero no significativas con 36 ventanas.

## 4. Test de Friedman + post-hoc de Nemenyi

El test de Friedman contrasta si los rangos medios de los tres modelos difieren sobre las 36 ventanas (bloques). Si se rechaza la nula, el post-hoc de Nemenyi identifica qué pares difieren. Se implementa el Nemenyi con la distribución del rango studentizado (equivalente a `scikit_posthocs.posthoc_nemenyi_friedman`).

In [5]:
def nemenyi(data):
    """data: matriz (n bloques × k modelos). Devuelve (rangos medios, matriz de p-valores)."""
    n, k = data.shape
    ranks = np.apply_along_axis(stats.rankdata, 1, data)
    mean_ranks = ranks.mean(axis=0)
    se = math.sqrt(k * (k + 1) / (6.0 * n))
    pmat = np.ones((k, k))
    for i in range(k):
        for j in range(i + 1, k):
            q = abs(mean_ranks[i] - mean_ranks[j]) / se * math.sqrt(2)
            pmat[i, j] = pmat[j, i] = stats.studentized_range.sf(q, k, np.inf)
    return mean_ranks, pmat

fams3 = ["XGBoost", "Chronos-2", "TTM"]
fried_rows, nem_frames = [], []
for H in HS:
    mat = np.array([err[(err.modelo == best_cfg[(f, H)][0]) &
                        (err.escenario == best_cfg[(f, H)][1]) &
                        (err.H == H)].sort_values("ventana")["MAE"].values
                    for f in fams3]).T                      # (36, 3)
    chi2, p_f = stats.friedmanchisquare(*mat.T)
    mean_ranks, pmat = nemenyi(mat)
    fried_rows.append(dict(H=H, chi2_friedman=chi2, p_friedman=p_f,
                           **{f"rango_medio_{f}": r for f, r in zip(fams3, mean_ranks)}))
    nem = pd.DataFrame(pmat, index=fams3, columns=fams3)
    nem.insert(0, "H", H)
    nem_frames.append(nem.reset_index().rename(columns={"index": "modelo"}))

fried_df = pd.DataFrame(fried_rows)
nem_df = pd.concat(nem_frames, ignore_index=True)
fried_df.to_csv("results/friedman.csv", index=False)
nem_df.to_csv("results/nemenyi.csv", index=False)
fried_df.round(4)

Out[0]: 
    H  chi2_friedman  ...  rango_medio_Chronos-2  rango_medio_TTM
0   6         0.3889  ...                 2.0833           1.9722
1  12         0.7222  ...                 2.1111           1.9167
2  24         4.2222  ...                 2.2778           1.8333
3  48         0.7222  ...                 2.0278           1.8889

[4 rows x 6 columns]


In [6]:
nem_df.round(4)

Out[0]: 
       modelo   H  XGBoost  Chronos-2     TTM
0     XGBoost   6   1.0000     0.8259  0.9924
1   Chronos-2   6   0.8259     1.0000  0.8847
2         TTM   6   0.9924     0.8847  1.0000
3     XGBoost  12   1.0000     0.8259  0.9698
4   Chronos-2  12   0.8259     1.0000  0.6875
5         TTM  12   0.9698     0.6875  1.0000
6     XGBoost  24   1.0000     0.2248  0.9698
7   Chronos-2  24   0.2248     1.0000  0.1428
8         TTM  24   0.9698     0.1428  1.0000
9     XGBoost  48   1.0000     0.9698  0.6875
10  Chronos-2  48   0.9698     1.0000  0.8259
11        TTM  48   0.6875     0.8259  1.0000


**Lectura**: el test de Friedman no rechaza la igualdad de rangos en ningún horizonte (p ≥ 0,12); en consecuencia, el post-hoc de Nemenyi tampoco señala pares significativos. Los rangos medios son casi idénticos (≈ 2,0 para los tres modelos), lo que indica que ningún modelo domina ventana a ventana de forma consistente: las diferencias agregadas de MAE provienen de subconjuntos de ventanas, no de una superioridad uniforme.

## 5. Coste computacional

Se combina `coste_computacional.csv` con el MAE a H = 48 de cada configuración. Notas:
- **XGBoost** no tiene «parámetros» en el sentido de una red neuronal; como orden de magnitud se usa el número de nodos del conjunto de árboles del modelo E2 entrenado (≈ 9·10³ nodos, 300 árboles).
- **Chronos-2**: 119,5 M de parámetros; el fine-tuning LoRA cabe en los 8 GB de la GPU (pico ≈ 2,3 GB).
- **TTM**: ≈ 0,8 M de parámetros (142–148× menos que Chronos-2 según la variante: 805.280 parámetros en zero-shot, 842.848 con covariables), inferencia por ventana < 0,5 ms incluso en CPU.

In [7]:
mae48 = cons[cons.H == 48][["modelo", "escenario", "MAE"]].rename(columns={"MAE": "MAE_H48"})
cm = coste.merge(mae48, on=["modelo", "escenario"], how="left")
cm["familia"] = cm["modelo"].map(FAM)
cm["t_inf_s"] = cm["t_inf_ventana_gpu_s"].fillna(cm["t_inf_ventana_cpu_s"])
cm["t_total_36v_s"] = cm["t_train_s"] + 36 * cm["t_inf_s"]   # coste total del experimento por config
out = cm[["familia", "modelo", "escenario", "n_parametros", "dispositivo",
          "t_train_s", "t_inf_s", "mem_gpu_pico_MB", "MAE_H48", "t_total_36v_s"]]
out.to_csv("results/coste_analisis.csv", index=False)
out.round(4)

Out[0]: 
      familia    modelo  ... MAE_H48  t_total_36v_s
0     XGBoost   xgboost  ...  4.3884         0.5114
1     XGBoost   xgboost  ...  4.3116         0.5542
2     XGBoost   xgboost  ...  4.3722         0.6192
3   Chronos-2  chronos2  ...  4.5345         0.0589
4   Chronos-2  chronos2  ...  4.4074         0.7051
5   Chronos-2  chronos2  ...  4.1977         1.0021
6   Chronos-2  chronos2  ...  4.4741        51.4151
7   Chronos-2  chronos2  ...  4.4256        55.6503
8   Chronos-2  chronos2  ...  4.2299        52.6736
9         TTM       ttm  ...  4.4237         0.0110
10        TTM       ttm  ...  4.3298         1.0888
11        TTM       ttm  ...  4.3300         3.0801
12        TTM       ttm  ...  4.2853        10.1520

[13 rows x 10 columns]


**Lectura**:
- El coste total de evaluar una configuración (entrenamiento + 36 ventanas) va de **11 ms** (TTM zero-shot) a **≈ 56 s** (Chronos-2 con LoRA). Todos los costes son perfectamente asumibles en una GPU de consumo (RTX 3070, 8 GB).
- El pico de VRAM del fine-tuning LoRA de Chronos-2 (≈ 2,3 GB) confirma que es **viable en 8 GB**, aunque, como muestran las secciones 1–4, no supera al zero-shot con covariables futuras.
- TTM ofrece la mejor relación precisión/coste a H = 48: MAE 4,29 con 0,8 M de parámetros y ~0,4 ms por ventana.

### Ficheros generados
- `results/tabla_consolidada.csv` · `results/mejor_por_familia.csv` · `results/diebold_mariano.csv` · `results/friedman.csv` · `results/nemenyi.csv` · `results/coste_analisis.csv`